# Fairness Metrics for Predictive Models

## Requirements

In [1]:
from aif360.datasets import GermanDataset
from aif360.sklearn.metrics import statistical_parity_difference, disparate_impact_ratio, equal_opportunity_difference, average_odds_error

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import pandas as pd 
import numpy as np

from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'
pip install 'aif360[OptimalTransport]'


## Initial elements

### Data


In [2]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [3]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0



### Response and predictors


In [4]:
quant_predictors = ['month', 'credit_amount', 'investment_as_income_percentage',
                    'residence_since', 'number_of_credits', 'people_liable_for']
cat_predictors = [x for x in german_data_pd.columns if x not in quant_predictors + [response_name]]
predictors = quant_predictors + cat_predictors

In [5]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute


### Outer evaluation method: simple-validation (train-test split)

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)

### Fairness metrics


In [7]:
logistic_reg = LogisticRegression(solver='liblinear', random_state=123)
logistic_reg.fit(X_train, Y_train)
Y_test_hat = logistic_reg.predict(X_test)
A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set


#### Statistical parity difference (independence)


In [8]:
value = statistical_parity_difference(y_true=Y_test,
                                      y_pred=Y_test_hat, 
                                      prot_attr=A_test,
                                      priv_group=sens_privileged_groups[0][sens_variable_name], 
                                      pos_label=response_favorable_label)
value = float(value)
value

-0.32539682539682535

`statistical_parity_difference`  $\hspace{0.05cm}=\hspace{0.05cm} P\left(\widehat{Y} = + \hspace{0.1cm}|\hspace{0.1cm} A = \text{unprivileged}\right) - P\left(\widehat{Y} = + \hspace{0.1cm}|\hspace{0.1cm} A = \text{privileged}\right)$

**Properties**

- $\in [-1, 1]$

- The closer to $-1$, the more favorable are the results (predictions) of the models to the privileged class of the sensitive variable.

- The closer to $1$, the more favorable are the results (predictions) of the models to the unprivileged class of the sensitive variable.

- The closer to $0$, the more fairness (parity) in the model results (predictions).


**Doubts**

- No veo que esta metrica sea una medida de justicia en todos los casos. Ejemplo: concesion de creditos, variable sensible color de piel (blanco, negro). Imaginemos que el grupo de los negros reune una serie de condiciones que les hace objetivamente mas riesgosos que los blancos (peor situacion economica, mayores deudas etc). Que en este caso el modelo conceda significativamente mas creditos a los blancos que a los negros no seria injusto, no? Puede setar basado en criterios economicos objetivos que estan asociados a esos grupos, y no por el color de piel en si, no??


---


#### Absolute statistical parity difference (absolute independence)


In [9]:
def abs_statistical_parity_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    value = statistical_parity_difference(y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label)
    return float(np.abs(value))

In [10]:
value = abs_statistical_parity_difference(y_true=Y_test,
                                          y_pred=Y_test_hat, 
                                          prot_attr=A_test,
                                          priv_group=sens_privileged_groups[0][sens_variable_name], 
                                          pos_label=response_favorable_label)
value

0.32539682539682535

`abs_statistical_parity_difference`  $\hspace{0.05cm}=\hspace{0.05cm} \left| P\left(\widehat{Y} = + \hspace{0.1cm}|\hspace{0.1cm} A = \text{unprivileged}\right) - P\left(\widehat{Y} = + \hspace{0.1cm}|\hspace{0.1cm} A = \text{privileged}\right) \right|$

- $P(\hat{Y} = + \hspace{0.08cm} | \hspace{0.08cm} A = \text{unprivileged})= \text{Proportion of credit granted to black people}$

- $P(\hat{Y} = + \hspace{0.08cm} | \hspace{0.08cm} A = \text{privileged})= \text{Proportion of credit granted to white people}$

**Properties**

- $\in [0,1]$
- The **closer to $0$** the less parity difference between the sensitive groups (**more fairness** (?)).
- The **closer to $1$** the more parity difference between the sensitive groups (**less fairness** (?)).


---


#### Disparate impact ratio


In [11]:
value = disparate_impact_ratio(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test,
                               priv_group=sens_privileged_groups[0][sens_variable_name], 
                               pos_label=response_favorable_label)
value

0.6057692307692308

  `disparate_impact_ratio` $\hspace{0.05cm}=\hspace{0.05cm}\dfrac{P(\hat{Y} =+ \hspace{0.08cm} |\hspace{0.08cm} A = \text{unprivileged})}
{Pr(\hat{Y} = + \hspace{0.08cm}|\hspace{0.08cm} A = \text{privileged})}$

- $P(\hat{Y} = + \hspace{0.08cm}|\hspace{0.08cm} A = \text{unprivileged})= \text{Proportion of credit granted to black people}$

- $P(\hat{Y} = + \hspace{0.08cm}|\hspace{0.08cm} A = \text{privileged})= \text{Proportion of credit granted to white people}$

**Properties**

- $\in [0,\infty]$
- The **closer to $1$**, the less parity difference between the sensitive groups (**more fairness**).
- The **closer to $0$ or to $\infty$** (**farther from $1$**) , the more parity difference between the sensitive groups (**less fairness**).

---

#### Absolute equal opportunity difference

In [12]:
def abs_equal_opportunity_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    value = equal_opportunity_difference(y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label)
    return float(np.abs(value))

In [13]:
abs_equal_opportunity_difference(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test, 
                             priv_group=sens_privileged_groups[0][sens_variable_name], 
                             pos_label=response_favorable_label)


0.320855614973262

`equal_opportunity_difference` $\hspace{0.1cm}=\hspace{0.1cm} \left| \hspace{0.05cm} P( \hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=+\hspace{0.04cm},\hspace{0.08cm} A = \text{unprivilaged}) \hspace{0.1cm}-\hspace{0.1cm} P( \hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm}  Y=+\hspace{0.04cm},\hspace{0.08cm}  A = \text{privilaged}) \hspace{0.05cm}\right| \\[0.5cm] \hspace{4.85cm} = \Big| \hspace{0.05cm} TPR_{A=\text{unprivilaged}}\hspace{0.1cm}-\hspace{0.1cm} TPR_{A=\text{privilaged}} \hspace{0.05cm} \Big|$

- $P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=+\hspace{0.04cm},\hspace{0.08cm} A = \text{unprivilaged}) = \text{Proportion of credits granted by the model (predicted positive) among black people (unprivileged) that actually deserved it (true positive)} = TPR_{A=\text{unprivilaged}}$

- $P( \hat{Y}=+\hspace{0.1cm}|\hspace{0.1cm} Y=+\hspace{0.04cm},\hspace{0.08cm} A = \text{unprivilaged}) = \text{Proportion of credit granted by the model  (predicted positive) among white people (privileged) that actually deserved it (true positive)} = TPR_{A=\text{privilaged}}$

**Properties**

- $\in [0, 1]$
- The **closer to $0$**, the less difference of opportunity (e.g. for getting the credit) between the sensitive groups (**more fairness**).
- The **closer to $1$**, the more difference of opportunity (e.g. for getting the credit) between the sensitive groups (**less fairness**).


---

#### Average odds error

In [14]:
def average_odds_error(y_true, y_pred, prot_attr, priv_group, pos_label):
    value = average_odds_error(y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label)
    return float(value)

In [15]:
average_odds_error(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test, 
                     priv_group=sens_privileged_groups[0][sens_variable_name], 
                     pos_label=response_favorable_label)

0.3333601383136987

`average_fpr_tpr_error` $\hspace{0.1cm}=\hspace{0.1cm} \dfrac{|P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=- \hspace{0.08cm},\hspace{0.08cm} A = \text{unpriv})  - P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=- \hspace{0.08cm},\hspace{0.08cm} A = \text{priv})| + |P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=+\hspace{0.08cm},\hspace{0.08cm} A = \text{unpriv})  - P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=+ \hspace{0.08cm},\hspace{0.08cm} A  = \text{priv}) |}{2} \\[0.3cm] \hspace{3.15cm} = \hspace{0.1cm} \dfrac{|FPR_{A = \text{unpriv}} - FPR_{A = \text{priv}}| + |TPR_{A = \text{unpriv}} - TPR_{A = \text{priv}}|}{2}$

**Properties**

- $\in [0, 1]$
- The **closer to $0$**, the less differences in how the model treats the sensitive groups (**more fairness**).
- The **closer to $1$**, the more differences in how the model treats the sensitive groups (**less fairness**).


---

#### Separation

In [ ]:
def false_positive_rate_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    fpr_unpriv = 

In [23]:
y_true = Y_test.to_numpy()
y_pred = Y_test_hat 
prot_attr = A_test.to_numpy()
priv_group=sens_privileged_groups[0][sens_variable_name] 
pos_label=response_favorable_label

In [41]:
neg_label_list = [int(x) for x in np.unique(y_true) if x != pos_label]
neg_label = neg_label_list if len(np.unique(y_true)) > 2 else neg_label_list[0]
neg_label

0

In [44]:
true_positive_idx = np.where(y_true == pos_label)[0]

In [42]:
predicted_negative_idx = np.where(y_pred == neg_label)[0]

In [45]:
true_positive_idx

array([  0,   1,   2,   6,   7,   8,  10,  11,  14,  16,  18,  19,  21,
        22,  23,  25,  26,  30,  31,  32,  33,  35,  36,  37,  39,  40,
        41,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  54,  56,
        57,  59,  60,  62,  63,  65,  66,  69,  71,  72,  74,  75,  76,
        78,  79,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        93,  94,  95,  96,  98, 100, 101, 102, 104, 105, 106, 107, 110,
       111, 112, 113, 115, 116, 118, 119, 121, 123, 124, 125, 126, 128,
       130, 131, 133, 136, 138, 139, 140, 141, 143, 145, 146, 147, 148,
       149])

In [46]:
predicted_negative_idx

array([  5,   6,   9,  12,  13,  14,  20,  21,  22,  24,  28,  29,  30,
        34,  38,  47,  55,  58,  61,  68,  73,  76,  88,  91, 101, 103,
       105, 108, 114, 119, 129, 141, 143, 149])

In [50]:
n_true_positive_pred_negative = np.sum([x in predicted_negative_idx for x in true_positive_idx])
n_true_positive = len(true_positive_idx)
n_true_positive_pred_negative /  n_true_positive

np.float64(0.14285714285714285)

In [49]:
np.sum(np.array([1,2]))

np.int64(3)

In [ ]:
def false_negative_rate_difference(y_true, y_pred, prot_attr, priv_group, pos_label)